# 01. Первичный анализ данных

Цель блокнота — понять устройство датасета до построения алгоритма кандидатогенерации.
Здесь проверяются размеры таблиц, единица наблюдения, повторяемость запросов.


## 1. Загрузка данных


In [ ]:
from pathlib import Path

import numpy as np

import pandas as pd


In [ ]:

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "dataset"

train_path = DATA_DIR / "train.parquet"
benchmark_queries_path = DATA_DIR / "benchmark_queries.parquet"
benchmark_items_path = DATA_DIR / "benchmark_items.parquet"


In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

In [6]:
train = pd.read_parquet(train_path)
benchmark_queries = pd.read_parquet(benchmark_queries_path)
benchmark_items = pd.read_parquet(benchmark_items_path)

train.head()


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлицами и ИП Опыт работы ...",e8b685dffe1a408e,"Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% от рыночной стоимости.\nВсе пр...",114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха Тип стоимости за ус...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем Новгороде и области.\n\n🫴 Помогу подобрать оптимальный вариа...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое товарищество Веретен...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтобы остановиться: выключить телефон, отложить дела и подарить ...",114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услуги Производство, обработка Место оказания услуг Краснодарский край, Ка...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу на официальных основаниях. \r\nЛЮБОЙ РЕГИОН России 🇷🇺.\r\nИзготовление...,114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,114,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ключ Место оказания услуг ул. Полины Осипенко Тип ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114


## 2. Как выглядит одна обучающая строка

Одна строка `train` — это зафиксированное взаимодействие: пользователь сформировал
поисковый контекст (`search_*`) и выбрал объявление (`item_*`)

In [11]:
train.iloc[0]


search_query                                                                                                                                                                                                                                                                                                                                                                    скупка телевизоров
search_location_id                                                                                                                                                                                                                                                                                                                                                                          652430
search_is_delivery_search                                                                                                                                                                                                         

## 3. Размер и разнообразие данных

Сначала оценим масштаб задачи и частотность текстов запросов.


In [12]:
train.shape

(497673, 19)

In [13]:
train["search_query"].nunique()

74529

In [14]:
train["item_id"].nunique()

344825

In [15]:
train["search_query"].value_counts().head(20)

search_query
маникюр                    6540
массаж                     5702
наращивание ресниц         5395
установка кондиционеров    4573
педикюр                    3436
сантехник                  3405
вывоз мусора               3278
эвакуатор                  3263
электрик                   3158
автоэлектрик               2790
покос травы                2477
вспашка земли              2315
манипулятор                2120
грузоперевозки газель      2034
натяжные потолки           2004
наращивание ногтей         1902
торты на заказ             1682
тонировка                  1644
поклейка обоев             1611
грузчики                   1540
Name: count, dtype: int64

### Краткая сводка

| Набор / показатель | Значение |
|---|---:|
| Строк в `train` | 497 673 |
| Уникальных текстов `search_query` | 74 529 |
| Уникальных `item_id` в `train` | 344 825 |
| Запросов в benchmark | 2 452 |
| Объявлений в benchmark-корпусе | 189 212 |




## 4. Что считать одним запросом

Одинаковый текст запроса может относиться к другой локации, категории, доставке или
набору фильтров. Поэтому для анализа релевантных объявлений используем полный
поисковый контекст из всех колонок `search_*`, а не только `search_query`.


In [16]:
search_columns = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
query_counts = train.groupby(
    search_columns,
    dropna=False,
).size()

In [18]:
query_counts.describe()


count    354463.000000
mean          1.404020
std           2.607192
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         628.000000
dtype: float64

In [19]:
(query_counts == 1).mean()

np.float64(0.8359038884171267)

In [20]:
relevant_items_per_query = (
    train.groupby(search_columns, dropna=False)["item_id"]
    .nunique()
)

In [21]:
relevant_items_per_query.describe()

count    354463.000000
mean          1.317638
std           1.611180
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         278.000000
Name: item_id, dtype: float64

In [22]:
(relevant_items_per_query == 1).mean()

np.float64(0.8500097330327848)

### Вывод по взаимодействиям

| Показатель для полного поискового контекста | Значение |
|---|---:|
| Уникальных контекстов | 354 463 |
| Среднее число строк на контекст | 1.40 |
| Контекстов ровно с одной строкой | 83.6% |
| Среднее число уникальных релевантных объявлений | 1.32 |
| Контекстов ровно с одним релевантным объявлением | 85.0% |



## 5. Связь train с benchmark

Проверяем две разные вещи: встречался ли **текст запроса** раньше и встречалось ли
**объявление из целевого корпуса** в обучающих взаимодействиях.


In [24]:
benchmark_queries.shape, benchmark_items.shape

((2452, 6), (189212, 14))

In [25]:
benchmark_queries["search_query"].isin(
    train["search_query"]
).mean()

np.float64(0.3699021207177814)

In [26]:
train["item_id"].isin(
    benchmark_items["item_id"]
).mean()

np.float64(0.06632869374066908)

In [29]:
train_item_ids = set(train["item_id"])
benchmark_item_ids = set(benchmark_items["item_id"])

common_item_ids = train_item_ids & benchmark_item_ids
print(len(common_item_ids), len(common_item_ids) / len(benchmark_item_ids), len(common_item_ids) / len(train_item_ids))

18142 0.09588186795763483 0.05261219459145944


### Сводка пересечений

| Проверка | Результат |
|---|---:|
| Benchmark-запросов с текстом, встречавшимся в train | 37.0% |
| Строк train, чьи `item_id` есть в benchmark-корпусе | 6.63% |
| Общих уникальных `item_id` | 18 142 |
| Доля benchmark-корпуса, встречавшаяся в train | 9.59% |
| Доля train-объявлений, вошедшая в benchmark-корпус | 5.26% |

Нельзя строить решение только на запоминании пар из train: большая часть запросов
и объявлений требует обобщения по тексту и признакам.


## 6. Итоги

1. Единица запроса — полный набор `search_*`, а не только текст.
2. Разметка положительная и неполная, поэтому основная метрика — `Recall@50`.
3. Точное запоминание ограничено: пересечение с целевым корпусом небольшое.
4. Для retrieval нужны признаки, способные обобщать: текст, параметры, категория и география.

